# Skin Lesion Diagnosis Full Training Pipeline

## Environment Setup & Imports

In [ ]:
import os
import sys
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import CFG, seed_everything, get_device
from src.dataset import (
    build_splits, get_dataloaders, compute_class_weights,
    SkinLesionDataset, get_val_transforms,
)
from src.models import build_model
from src.train import train_model
from src.evaluate import (
    evaluate_model, build_comparison_table,
    plot_training_history, plot_confusion_matrix,
)
from src.gradcam import generate_gradcam_examples
from src.embeddings import run_embedding_analysis
from src.ablations import (
    study_simplecnn_augmentation, study_simplecnn_class_weights, study_simplecnn_depth,
    study_resnet50_augmentation, study_resnet50_class_weights, study_resnet50_freezing,
    study_vit_augmentation, study_vit_class_weights, study_vit_lora_rank,
    build_ablation_summary,
)

print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"Project root: {PROJECT_ROOT}")

## Configuration & Device

In [ ]:
device      = get_device(CFG)
results_dir = CFG.paths.results

seed_everything(CFG.project.random_seed)
torch.set_float32_matmul_precision("high")

for path in [
    results_dir,
    CFG.paths.checkpoints,
    f"{results_dir}/gradcam",
    f"{results_dir}/embeddings",
    f"{results_dir}/ablations",
    f"{results_dir}/plots",
]:
    os.makedirs(path, exist_ok=True)

print(f"Results dir: {results_dir}")

## Model Registry

Central source of truth: `(model_key, display_name, checkpoint_subpath)`

Change `SELECTED_MODEL` to train only one model, or leave as `"all"` for the full pipeline.

In [ ]:
MODEL_REGISTRY = [
    ("simple_cnn",  "Simple CNN",      "simple_cnn/best.pth"),
    ("resnet50",    "ResNet50",        "resnet50/best.pth"),
    ("vit",         "ViT-B/16 + LoRA", "vit/best.pth"),
]

SELECTED_MODEL = "all"   # options: "all", "resnet50", "vit", "simple_cnn"
RUN_ABLATIONS  = True    # set False to skip ablation studies

print(f"Selected model : {SELECTED_MODEL}")
print(f"Run ablations  : {RUN_ABLATIONS}")

## Data Splits & Dataloaders

In [ ]:
metadata_csv = f"{CFG.paths.data_raw}/HAM10000_metadata.csv"
train_df, val_df, test_df = build_splits(metadata_csv, CFG.paths.data_raw)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
class_weights = compute_class_weights(train_df)
train_loader, val_loader, test_loader = get_dataloaders(
    train_df, val_df, test_df, use_weighted_sampler=True,
)

print(f"Class weights: {class_weights}")

## Helper Functions

In [ ]:
def ckpt_path(subpath: str) -> str:
    return os.path.join(CFG.paths.checkpoints, subpath)


def load_checkpoint(model: torch.nn.Module, subpath: str) -> torch.nn.Module:
    # weights_only=True avoids arbitrary code execution from pickle payloads
    model.load_state_dict(torch.load(ckpt_path(subpath), weights_only=True))
    return model


def train_one(model_key, model, train_loader, val_loader, class_weights, device):
    return train_model(model, train_loader, val_loader, class_weights, device, model_name=model_key)

## Train Models

In [ ]:
all_metrics      = {}
all_histories    = {}
all_eval_results = {}
trained_models   = {}

for model_key, display_name, ckpt_subpath in MODEL_REGISTRY:
    if SELECTED_MODEL not in ("all", model_key):
        continue

    model = build_model(model_key, device)

    history = train_one(model_key, model, train_loader, val_loader, class_weights, device)
    all_histories[model_key] = history

    model = load_checkpoint(model, ckpt_subpath)
    eval_results = evaluate_model(model, test_loader, device,
                                  save_dir=results_dir, model_name=model_key)

    all_metrics[display_name]   = eval_results["metrics"]
    all_eval_results[model_key] = eval_results
    trained_models[model_key]   = model
    print(f"[{model_key}] Metrics: {eval_results['metrics']}")

## Training History Plots

Loss, Balanced Accuracy, Macro F1, Macro Precision, and ROC-AUC curves for each trained model.

### Simple CNN

In [ ]:
if "simple_cnn" in all_histories:
    plot_training_history(all_histories["simple_cnn"], f"{results_dir}/plots", "simple_cnn")

In [ ]:
if "simple_cnn" in all_eval_results:
    r = all_eval_results["simple_cnn"]
    plot_confusion_matrix(r["ground_truth"], r["predictions"], f"{results_dir}/plots", "simple_cnn")

### ResNet50

In [ ]:
if "resnet50" in all_histories:
    plot_training_history(all_histories["resnet50"], f"{results_dir}/plots", "resnet50")

In [ ]:
if "resnet50" in all_eval_results:
    r = all_eval_results["resnet50"]
    plot_confusion_matrix(r["ground_truth"], r["predictions"], f"{results_dir}/plots", "resnet50")

### ViT-B/16 + LoRA

In [ ]:
if "vit" in all_histories:
    plot_training_history(all_histories["vit"], f"{results_dir}/plots", "vit")

In [ ]:
if "vit" in all_eval_results:
    r = all_eval_results["vit"]
    plot_confusion_matrix(r["ground_truth"], r["predictions"], f"{results_dir}/plots", "vit")

## Embedding Analysis (PCA + t-SNE)

In [ ]:
for model_key, model in trained_models.items():
    print(f"[embeddings] {model_key}")
    run_embedding_analysis(model, test_loader, device,
                           save_dir=f"{results_dir}/embeddings",
                           model_name=model_key)

## GradCAM Visualisations

GradCAM requires a conv layer handle via `get_gradcam_layer()` (CNN-based models only).

In [ ]:
for gcam_model_key in ("resnet50", "simple_cnn"):
    if gcam_model_key in trained_models:
        model = trained_models[gcam_model_key]
        gradcam_model = getattr(model, "_orig_mod", model)
        test_ds = SkinLesionDataset(test_df, transform=get_val_transforms(), return_path=True)
        generate_gradcam_examples(
            gradcam_model, gradcam_model.get_gradcam_layer(), test_ds, device,
            save_dir=f"{results_dir}/gradcam", model_name=gcam_model_key,
            n_correct=8, n_wrong=8,
        )
        print(f"[gradcam] {gcam_model_key} done.")
    else:
        print(f"[gradcam] {gcam_model_key} not trained — skipping.")

## Model Comparison Table

In [ ]:
if len(all_metrics) > 1:
    build_comparison_table(all_metrics, results_dir)
    print("[comparison] Table saved.")
else:
    print("[comparison] Only one model trained — skipping comparison table.")

## Ablation Studies

Three studies per model: augmentation strategies, class weighting, and architecture depth / LoRA rank.

### Simple CNN — Ablation Studies

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML, Image
    import pandas as pd
    ablation_dir = f"{results_dir}/ablations"
    _scnn_abl = {}

    display(HTML("<h3>SimpleCNN Study 1: Data Augmentation</h3>"))
    s1 = study_simplecnn_augmentation(train_df, val_df, device, ablation_dir)
    _scnn_abl.update(s1)
    display(pd.DataFrame([
        {"Setting": "No augmentation",    "Best Val Acc": f"{s1['scnn_no_aug']['best_val_acc']:.4f}",   "Best Val F1": f"{s1['scnn_no_aug']['best_val_f1']:.4f}",   "Best Val Prec": f"{s1['scnn_no_aug']['best_val_precision']:.4f}",   "Best Val AUC": f"{s1['scnn_no_aug']['best_val_roc_auc']:.4f}"},
        {"Setting": "With augmentation ✓","Best Val Acc": f"{s1['scnn_with_aug']['best_val_acc']:.4f}", "Best Val F1": f"{s1['scnn_with_aug']['best_val_f1']:.4f}", "Best Val Prec": f"{s1['scnn_with_aug']['best_val_precision']:.4f}", "Best Val AUC": f"{s1['scnn_with_aug']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))
else:
    print("[ablations] Simple CNN skipped.")

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML, Image
    ablation_dir = f"{results_dir}/ablations"
    display(HTML("<h4>Study 1 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/simplecnn_study1_augmentation.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: No Augmentation</h4>"))
    display(Image(f"{ablation_dir}/scnn_aug_off_confusion_matrix.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: With Augmentation</h4>"))
    display(Image(f"{ablation_dir}/scnn_aug_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>SimpleCNN Study 2: Class Weighting</h3>"))
    s2 = study_simplecnn_class_weights(train_df, val_df, device, ablation_dir)
    _scnn_abl.update(s2)
    display(pd.DataFrame([
        {"Setting": "Uniform loss",       "Best Val Acc": f"{s2['scnn_no_weights']['best_val_acc']:.4f}",   "Best Val F1": f"{s2['scnn_no_weights']['best_val_f1']:.4f}",   "Best Val Prec": f"{s2['scnn_no_weights']['best_val_precision']:.4f}",   "Best Val AUC": f"{s2['scnn_no_weights']['best_val_roc_auc']:.4f}"},
        {"Setting": "Weighted CE loss ✓", "Best Val Acc": f"{s2['scnn_with_weights']['best_val_acc']:.4f}", "Best Val F1": f"{s2['scnn_with_weights']['best_val_f1']:.4f}", "Best Val Prec": f"{s2['scnn_with_weights']['best_val_precision']:.4f}", "Best Val AUC": f"{s2['scnn_with_weights']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 2 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/simplecnn_study2_class_weights.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Uniform Loss</h4>"))
    display(Image(f"{ablation_dir}/scnn_weights_off_confusion_matrix.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Weighted CE Loss</h4>"))
    display(Image(f"{ablation_dir}/scnn_weights_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>SimpleCNN Study 3: Network Depth</h3>"))
    s3 = study_simplecnn_depth(train_df, val_df, device, ablation_dir)
    _scnn_abl.update(s3)
    display(pd.DataFrame([
        {"Setting": "Shallow (2 blocks)", "Best Val Acc": f"{s3['scnn_shallow']['best_val_acc']:.4f}", "Best Val F1": f"{s3['scnn_shallow']['best_val_f1']:.4f}", "Best Val Prec": f"{s3['scnn_shallow']['best_val_precision']:.4f}", "Best Val AUC": f"{s3['scnn_shallow']['best_val_roc_auc']:.4f}"},
        {"Setting": "Deep (4 blocks) ✓",  "Best Val Acc": f"{s3['scnn_deep']['best_val_acc']:.4f}",   "Best Val F1": f"{s3['scnn_deep']['best_val_f1']:.4f}",   "Best Val Prec": f"{s3['scnn_deep']['best_val_precision']:.4f}",   "Best Val AUC": f"{s3['scnn_deep']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 3 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/simplecnn_study3_depth.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: Shallow (2 blocks)</h4>"))
    display(Image(f"{ablation_dir}/scnn_shallow_confusion_matrix.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: Deep (4 blocks)</h4>"))
    display(Image(f"{ablation_dir}/scnn_deep_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "simple_cnn"):
    from IPython.display import display, HTML
    display(HTML("<h3>SimpleCNN — Full Ablation Summary</h3>"))
    summary_df = build_ablation_summary(_scnn_abl, ablation_dir)
    display(summary_df)
    print("[ablations] Simple CNN done.")

### ResNet50 — Ablation Studies

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML, Image
    import pandas as pd
    ablation_dir = f"{results_dir}/ablations"
    _r50_abl = {}

    display(HTML("<h3>ResNet50 Study 1: Data Augmentation</h3>"))
    s1 = study_resnet50_augmentation(train_df, val_df, device, ablation_dir)
    _r50_abl.update(s1)
    display(pd.DataFrame([
        {"Setting": "No augmentation",    "Best Val Acc": f"{s1['r50_no_aug']['best_val_acc']:.4f}",   "Best Val F1": f"{s1['r50_no_aug']['best_val_f1']:.4f}",   "Best Val Prec": f"{s1['r50_no_aug']['best_val_precision']:.4f}",   "Best Val AUC": f"{s1['r50_no_aug']['best_val_roc_auc']:.4f}"},
        {"Setting": "With augmentation ✓","Best Val Acc": f"{s1['r50_with_aug']['best_val_acc']:.4f}", "Best Val F1": f"{s1['r50_with_aug']['best_val_f1']:.4f}", "Best Val Prec": f"{s1['r50_with_aug']['best_val_precision']:.4f}", "Best Val AUC": f"{s1['r50_with_aug']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))
else:
    print("[ablations] ResNet50 skipped.")

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 1 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/resnet50_study1_augmentation.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: No Augmentation</h4>"))
    display(Image(f"{ablation_dir}/r50_aug_off_confusion_matrix.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: With Augmentation</h4>"))
    display(Image(f"{ablation_dir}/r50_aug_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>ResNet50 Study 2: Class Weighting</h3>"))
    s2 = study_resnet50_class_weights(train_df, val_df, device, ablation_dir)
    _r50_abl.update(s2)
    display(pd.DataFrame([
        {"Setting": "Uniform loss",       "Best Val Acc": f"{s2['r50_no_weights']['best_val_acc']:.4f}",   "Best Val F1": f"{s2['r50_no_weights']['best_val_f1']:.4f}",   "Best Val Prec": f"{s2['r50_no_weights']['best_val_precision']:.4f}",   "Best Val AUC": f"{s2['r50_no_weights']['best_val_roc_auc']:.4f}"},
        {"Setting": "Weighted CE loss ✓", "Best Val Acc": f"{s2['r50_with_weights']['best_val_acc']:.4f}", "Best Val F1": f"{s2['r50_with_weights']['best_val_f1']:.4f}", "Best Val Prec": f"{s2['r50_with_weights']['best_val_precision']:.4f}", "Best Val AUC": f"{s2['r50_with_weights']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 2 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/resnet50_study2_class_weights.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Uniform Loss</h4>"))
    display(Image(f"{ablation_dir}/r50_weights_off_confusion_matrix.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Weighted CE Loss</h4>"))
    display(Image(f"{ablation_dir}/r50_weights_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>ResNet50 Study 3: Backbone Freezing</h3>"))
    s3 = study_resnet50_freezing(train_df, val_df, device, ablation_dir)
    _r50_abl.update(s3)
    display(pd.DataFrame([
        {"Setting": "Frozen backbone",    "Best Val Acc": f"{s3['r50_frozen']['best_val_acc']:.4f}",  "Best Val F1": f"{s3['r50_frozen']['best_val_f1']:.4f}",  "Best Val Prec": f"{s3['r50_frozen']['best_val_precision']:.4f}",  "Best Val AUC": f"{s3['r50_frozen']['best_val_roc_auc']:.4f}"},
        {"Setting": "Full fine-tuning ✓", "Best Val Acc": f"{s3['r50_full_ft']['best_val_acc']:.4f}", "Best Val F1": f"{s3['r50_full_ft']['best_val_f1']:.4f}", "Best Val Prec": f"{s3['r50_full_ft']['best_val_precision']:.4f}", "Best Val AUC": f"{s3['r50_full_ft']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 3 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/resnet50_study3_backbone.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: Frozen Backbone</h4>"))
    display(Image(f"{ablation_dir}/r50_frozen_confusion_matrix.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: Full Fine-tuning</h4>"))
    display(Image(f"{ablation_dir}/r50_full_ft_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "resnet50"):
    from IPython.display import display, HTML
    display(HTML("<h3>ResNet50 — Full Ablation Summary</h3>"))
    summary_df = build_ablation_summary(_r50_abl, ablation_dir)
    display(summary_df)
    print("[ablations] ResNet50 done.")

### ViT-B/16 + LoRA — Ablation Studies

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML, Image
    import pandas as pd
    ablation_dir = f"{results_dir}/ablations"
    _vit_abl = {}

    display(HTML("<h3>ViT+LoRA Study 1: Data Augmentation</h3>"))
    s1 = study_vit_augmentation(train_df, val_df, device, ablation_dir)
    _vit_abl.update(s1)
    display(pd.DataFrame([
        {"Setting": "No augmentation",    "Best Val Acc": f"{s1['vit_no_aug']['best_val_acc']:.4f}",   "Best Val F1": f"{s1['vit_no_aug']['best_val_f1']:.4f}",   "Best Val Prec": f"{s1['vit_no_aug']['best_val_precision']:.4f}",   "Best Val AUC": f"{s1['vit_no_aug']['best_val_roc_auc']:.4f}"},
        {"Setting": "With augmentation ✓","Best Val Acc": f"{s1['vit_with_aug']['best_val_acc']:.4f}", "Best Val F1": f"{s1['vit_with_aug']['best_val_f1']:.4f}", "Best Val Prec": f"{s1['vit_with_aug']['best_val_precision']:.4f}", "Best Val AUC": f"{s1['vit_with_aug']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))
else:
    print("[ablations] ViT skipped.")

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 1 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/vit_study1_augmentation.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: No Augmentation</h4>"))
    display(Image(f"{ablation_dir}/vit_aug_off_confusion_matrix.png"))
    display(HTML("<h4>Study 1 — Confusion Matrix: With Augmentation</h4>"))
    display(Image(f"{ablation_dir}/vit_aug_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>ViT+LoRA Study 2: Class Weighting</h3>"))
    s2 = study_vit_class_weights(train_df, val_df, device, ablation_dir)
    _vit_abl.update(s2)
    display(pd.DataFrame([
        {"Setting": "Uniform loss",       "Best Val Acc": f"{s2['vit_no_weights']['best_val_acc']:.4f}",   "Best Val F1": f"{s2['vit_no_weights']['best_val_f1']:.4f}",   "Best Val Prec": f"{s2['vit_no_weights']['best_val_precision']:.4f}",   "Best Val AUC": f"{s2['vit_no_weights']['best_val_roc_auc']:.4f}"},
        {"Setting": "Weighted CE loss ✓", "Best Val Acc": f"{s2['vit_with_weights']['best_val_acc']:.4f}", "Best Val F1": f"{s2['vit_with_weights']['best_val_f1']:.4f}", "Best Val Prec": f"{s2['vit_with_weights']['best_val_precision']:.4f}", "Best Val AUC": f"{s2['vit_with_weights']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 2 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/vit_study2_class_weights.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Uniform Loss</h4>"))
    display(Image(f"{ablation_dir}/vit_weights_off_confusion_matrix.png"))
    display(HTML("<h4>Study 2 — Confusion Matrix: Weighted CE Loss</h4>"))
    display(Image(f"{ablation_dir}/vit_weights_on_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML
    import pandas as pd

    display(HTML("<h3>ViT+LoRA Study 3: LoRA Rank</h3>"))
    s3 = study_vit_lora_rank(train_df, val_df, device, ablation_dir)
    _vit_abl.update(s3)
    display(pd.DataFrame([
        {"Setting": "LoRA rank=2",   "Best Val Acc": f"{s3['vit_lora_rank2']['best_val_acc']:.4f}", "Best Val F1": f"{s3['vit_lora_rank2']['best_val_f1']:.4f}", "Best Val Prec": f"{s3['vit_lora_rank2']['best_val_precision']:.4f}", "Best Val AUC": f"{s3['vit_lora_rank2']['best_val_roc_auc']:.4f}"},
        {"Setting": "LoRA rank=8 ✓", "Best Val Acc": f"{s3['vit_lora_rank8']['best_val_acc']:.4f}", "Best Val F1": f"{s3['vit_lora_rank8']['best_val_f1']:.4f}", "Best Val Prec": f"{s3['vit_lora_rank8']['best_val_precision']:.4f}", "Best Val AUC": f"{s3['vit_lora_rank8']['best_val_roc_auc']:.4f}"},
    ]).set_index("Setting"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML, Image
    display(HTML("<h4>Study 3 — Comparison: Learning Curves &amp; Metrics</h4>"))
    display(Image(f"{ablation_dir}/vit_study3_lora_rank.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: LoRA rank=2</h4>"))
    display(Image(f"{ablation_dir}/vit_lora_rank2_confusion_matrix.png"))
    display(HTML("<h4>Study 3 — Confusion Matrix: LoRA rank=8</h4>"))
    display(Image(f"{ablation_dir}/vit_lora_rank8_confusion_matrix.png"))

In [ ]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "vit"):
    from IPython.display import display, HTML
    display(HTML("<h3>ViT+LoRA — Full Ablation Summary</h3>"))
    summary_df = build_ablation_summary(_vit_abl, ablation_dir)
    display(summary_df)
    print("[ablations] ViT done.")

## Pipeline Complete

In [ ]:
print(f"\n{'=' * 60}")
print("PIPELINE COMPLETE")
print(f"Results:   {results_dir}/")
print("Dashboard: streamlit run dashboard/app.py")
print("=" * 60)